# Scaled Dot-Product Attention in NumPy


In [1]:
import numpy as np

rng = np.random.default_rng(11)
d, T, d_v = 4, 6, 5
query = rng.normal(size=d)
keys = rng.normal(size=(T, d))
values = rng.normal(size=(T, d_v))
Q = rng.normal(size=(3, d))

def stable_softmax(x, axis=-1):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def dot_scores(query, keys):
    return keys @ query

def scaled_dot_scores(query, keys):
    d_k = keys.shape[-1]
    return dot_scores(query, keys) / np.sqrt(d_k)

def attention_weights(scores):
    return stable_softmax(scores)

def context_vector(weights, values):
    return weights @ values

def alignment_matrix(Q, K):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    return stable_softmax(scores, axis=1)

e = dot_scores(query, keys)
es = scaled_dot_scores(query, keys)
print(np.allclose(es, e / np.sqrt(4)))
w = attention_weights(es)
print(round(float(w.sum()), 6))
print(context_vector(w, values).shape)
Aln = alignment_matrix(Q, keys)
print(Aln.shape, np.allclose(Aln.sum(axis=1), 1))

rng2 = np.random.default_rng(3)
qh = rng2.normal(size=64)
Kh = rng2.normal(size=(8, 64))
ent = lambda p: -np.sum(p * np.log(p + 1e-12))
print(ent(attention_weights(scaled_dot_scores(qh, Kh))) >
      ent(attention_weights(dot_scores(qh, Kh))))


True
1.0
(5,)
(3, 6) True
True
